In [1]:
import pandas as pd
import numpy as np

import re, os

In [ ]:
df_ag = pd.read_csv('GSE70353_series_matrix.txt',sep='\t', comment='!').fillna('')
df_ngs = pd.read_csv('GSE135134_METSIM_subcutaneousAdipose_RNAseq_TPMs_n434.txt',sep='\t', comment='!').fillna('')

In [ ]:
df_ag_sample = pd.read_csv('GSE70353_series_matrix_head.txt',sep='\t',index_col=0).fillna('')
df_ag_sample = df_ag_sample.transpose()
df_ag_sample.head()
overlapped_samples = df_ag_sample[df_ag_sample['Sample_title'].isin(df_ngs.columns)] 
df_ngs['Ensembl'] = [re.sub(r'\.\d*','', x) for x in df_ngs['ID']]

In [ ]:
ag_anno = pd.read_csv('GPL13667-15572.txt',sep='\t', comment='#').fillna('')

In [ ]:
mapping_pool = {}
for _, d in name_map.iterrows():
    mapping_pool[d['Identifier c']]=d['Entrez gene id e']

In [ ]:
ag_idx, ngs_idx, gene_name=[], [], []
for ind, d in tqdm(df_ag.iterrows()):
    ensembl_id = ag_anno.loc[ind, 'Ensembl']
    if ensembl_id == '---': continue
    ag_count = ag_anno[ag_anno['Ensembl']==ensembl_id]
    ngs_count = df_ngs[df_ngs['Ensembl']==ensembl_id]
    if ag_count.shape[0]==1 and ngs_count.shape[0]==1:
        ag_idx.append(ind)
        ngs_idx.append(ngs_count.index.values[0])
        gene_name.append(ensembl_id)
df_ag_map = df_ag.loc[ag_idx,:]
df_ngs_map = df_ngs.loc[ngs_idx,:]        

In [ ]:
df_ngs_fin = df_ngs_map.drop(['ID','Ensembl'],axis=1)
df_ngs_fin = df_ngs_fin.transpose()
df_ag_fin = df_ag_map.transpose()

df_ag_fin = df_ag_fin.loc[overlapped_samples.index,:]
df_ngs_fin = df_ngs_fin.loc[overlapped_samples['Sample_title'],:]
df_ag_fin = df_ag_fin.set_index(df_ngs_fin.index)
print(df_ag_fin.shape, df_ngs_fin.shape)

In [ ]:
df_ngs_fin = np.log2(df_ngs_fin + 1)

In [ ]:
df_ag_fin.to_csv('df_ag.tsv',sep="\t")
df_ngs_fin.to_csv('df_ngs.tsv', sep="\t")